In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project

Mounted at /content/drive
/content/drive/MyDrive/ml_final_project


In [ ]:
from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline, split_and_save

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install prophet mlflow dagshub -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212

In [ ]:
import mlflow
import dagshub

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)
mlflow.set_experiment("XGBoost_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7d55bf82-7bf0-489e-b8c8-a507d97115b6&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=577e248bd91dfe05d6a88635a47c73daed4073456ec11f1b9981c907babe0be1




Accessing as aochi23

Initialized MLflow to track repo "aochi23/ml_final_project"

Repository aochi23/ml_final_project initialized!

<Experiment: artifact_location='mlflow-artifacts:/8e37b3bed1354d1cb17c47e5acd0874c', creation_time=1783526260953, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783526260953, lifecycle_stage='active', name='XGBoost_Training', tags={}, trace_location=None, workspace='default'>

In [ ]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')
full_df = run_feature_pipeline(train, test, features, stores)
processed_train, processed_test = split_and_save(full_df)

print(processed_train.shape)
processed_train.head()

(421570, 39)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,lag_52,lag_53,rolling_mean_4,rolling_mean_8,rolling_std_4,rolling_std_8,yoy_growth,dept_avg_sales,store_avg_sales,type_ordinal
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19213.485088,21710.543621,3
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,...,NaN,NaN,32990.77,NaN,12832.106391,NaN,NaN,19213.485088,21710.543621,3


In [ ]:
n_missing_lag52 = processed_train['lag_52'].isna().sum()
pct_missing_lag52 = n_missing_lag52 / len(processed_train) * 100
print(f"Rows missing lag_52: {n_missing_lag52} ({pct_missing_lag52:.1f}%)")

print(processed_train.loc[processed_train['lag_52'].isna(), 'Date'].agg(['min', 'max']))

n_sparse_pairs = processed_train.loc[processed_train['is_sparse'], ['Store','Dept']].drop_duplicates().shape[0]
n_total_pairs = processed_train[['Store','Dept']].drop_duplicates().shape[0]
print(f"Sparse pairs: {n_sparse_pairs} / {n_total_pairs}")

n_sparse_rows = processed_train['is_sparse'].sum()
pct_sparse_rows = n_sparse_rows / len(processed_train) * 100
print(f"Rows in sparse pairs: {n_sparse_rows} ({pct_sparse_rows:.1f}%)")

Rows missing lag_52: 160487 (38.1%)
min   2010-02-05
max   2012-10-26
Name: Date, dtype: datetime64[ns]
Sparse pairs: 340 / 3331
Rows in sparse pairs: 4955 (1.2%)


In [ ]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    mlflow.log_param("drop_missing_lag52_rows", False)
    mlflow.log_param("drop_sparse_pairs", False)
    mlflow.log_param("rationale", "lag52 missing=38.1% of rows -> too costly to drop; sparse pairs=1.2% of rows -> low impact either way, kept for test-time coverage")

    mlflow.log_metric("n_rows_total", len(processed_train))
    mlflow.log_metric("n_rows_missing_lag52", n_missing_lag52)
    mlflow.log_metric("pct_rows_missing_lag52", pct_missing_lag52)
    mlflow.log_metric("n_sparse_pairs", n_sparse_pairs)
    mlflow.log_metric("n_total_pairs", n_total_pairs)
    mlflow.log_metric("n_sparse_rows", n_sparse_rows)
    mlflow.log_metric("pct_sparse_rows", pct_sparse_rows)

🏃 View run XGBoost_Cleaning at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/e10c0c03e91741fcbe6fc819fd5e8166
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2


In [ ]:
print("Unique Stores:", processed_train['Store'].nunique())
print("Unique Depts:", processed_train['Dept'].nunique())
print("Store dtype:", processed_train['Store'].dtype)
print("Dept dtype:", processed_train['Dept'].dtype)

Unique Stores: 45
Unique Depts: 81
Store dtype: int64
Dept dtype: int64


In [ ]:
for df in [processed_train, processed_test]:
    df['Store'] = df['Store'].astype('category')
    df['Dept'] = df['Dept'].astype('category')

processed_train['yoy_growth_v2'] = (
    (processed_train['lag_52'] - processed_train['lag_53'])
    / processed_train['lag_53'].replace(0, np.nan)
)
processed_test['yoy_growth_v2'] = (
    (processed_test['lag_52'] - processed_test['lag_53'])
    / processed_test['lag_53'].replace(0, np.nan)
)

drop_cols = [
    'lag_1', 'lag_2', 'lag_3', 'lag_4',
    'rolling_mean_4', 'rolling_mean_8', 'rolling_std_4', 'rolling_std_8',
    'yoy_growth', 'Weekly_Sales','Date', 'Type',
]


feature_cols = [col for col in processed_train.columns if col not in drop_cols]
print(f"Final feature count: {len(feature_cols)}")
print(feature_cols)

Final feature count: 28
['Store', 'Dept', 'IsHoliday', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Size', 'markdown_total', 'n_active_markdowns', 'n_weeks', 'is_sparse', 'week_of_year', 'month', 'year', 'days_to_nearest_holiday', 'is_week_after_thanksgiving', 'lag_52', 'lag_53', 'dept_avg_sales', 'store_avg_sales', 'type_ordinal', 'yoy_growth_v2']


In [ ]:
with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("feature_cols", feature_cols)
    mlflow.log_param("dropped_cols", drop_cols)
    mlflow.log_param(
        "rationale",
        "Dropped short-horizon lags (lag_1-4, rolling_mean/std_4/8) since they are "
        "unavailable for most of the test horizon (only lag_52/lag_53 stay valid for the "
        "full 39-week test period). Rebuilt yoy_growth as yoy_growth_v2 using lag_52/lag_53 "
        "only. Dropped raw Type. Kept n_weeks alongside is_sparse"
    )

🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/54c7b13f6c55452ba8565db10add573f
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2


In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [ ]:
train_dates = sorted(processed_train['Date'].unique())
print(f"Train spans {train_dates[0]} to {train_dates[-1]}, {len(train_dates)} unique weeks")

Train spans 2010-02-05 00:00:00 to 2012-10-26 00:00:00, 143 unique weeks


In [ ]:
n_folds = 3
val_window_weeks = 8

train_dates = sorted(processed_train['Date'].unique())

folds = []
for i in range(n_folds):
    val_end_idx = len(train_dates) - i * val_window_weeks
    val_start_idx = val_end_idx - val_window_weeks
    val_dates = train_dates[val_start_idx:val_end_idx]
    train_dates_fold = train_dates[:val_start_idx]
    folds.append((train_dates_fold, val_dates))

for i, (train_dates_fold, val_dates) in enumerate(folds):
    print(f"Fold {i}: train {train_dates_fold[0]} to {train_dates_fold[-1]} "
          f"({len(train_dates_fold)} weeks), val {val_dates[0]} to {val_dates[-1]}")

Fold 0: train 2010-02-05 00:00:00 to 2012-08-31 00:00:00 (135 weeks), val 2012-09-07 00:00:00 to 2012-10-26 00:00:00
Fold 1: train 2010-02-05 00:00:00 to 2012-07-06 00:00:00 (127 weeks), val 2012-07-13 00:00:00 to 2012-08-31 00:00:00
Fold 2: train 2010-02-05 00:00:00 to 2012-05-11 00:00:00 (119 weeks), val 2012-05-18 00:00:00 to 2012-07-06 00:00:00


In [ ]:
fold_data = []

for i, (train_dates_fold, val_dates) in enumerate(folds):
    train_mask = processed_train['Date'].isin(train_dates_fold)
    val_mask = processed_train['Date'].isin(val_dates)

    X_train = processed_train.loc[train_mask, feature_cols]
    y_train = processed_train.loc[train_mask, 'Weekly_Sales']
    X_val = processed_train.loc[val_mask, feature_cols]
    y_val = processed_train.loc[val_mask, 'Weekly_Sales']
    holiday_val = processed_train.loc[val_mask, 'IsHoliday']

    fold_data.append((X_train, y_train, X_val, y_val, holiday_val))
    print(f"Fold {i}: X_train {X_train.shape}, X_val {X_val.shape}")

Fold 0: X_train (397841, 28), X_val (23729, 28)
Fold 1: X_train (374203, 28), X_val (23638, 28)
Fold 2: X_train (350595, 28), X_val (23608, 28)


In [ ]:
from xgboost import XGBRegressor

params = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
    "random_state": 42,
}

fold_scores = []

with mlflow.start_run(run_name="XGBoost_Baseline"):
    mlflow.log_params(params)
    mlflow.log_param("n_folds", n_folds)
    mlflow.log_param("val_window_weeks", val_window_weeks)

    for i, (X_train, y_train, X_val, y_val, holiday_val) in enumerate(fold_data):
        with mlflow.start_run(run_name=f"XGBoost_Baseline_fold{i}", nested=True):
            model = XGBRegressor(
                enable_categorical=True,
                tree_method='hist',
                **params,
            )
            model.fit(X_train, y_train)

            preds = model.predict(X_val)
            fold_wmae = wmae(y_val.values, preds, holiday_val.values)
            fold_scores.append(fold_wmae)

            mlflow.log_params(params)
            mlflow.log_metric("wmae", fold_wmae)
            mlflow.log_metric("n_train_rows", len(X_train))
            mlflow.log_metric("n_val_rows", len(X_val))

            print(f"Fold {i}: WMAE = {fold_wmae:.2f}")

    avg_wmae = np.mean(fold_scores)
    std_wmae = np.std(fold_scores)
    mlflow.log_metric("avg_wmae", avg_wmae)
    mlflow.log_metric("std_wmae", std_wmae)

    print(f"\nAverage WMAE across folds: {avg_wmae:.2f} (+/- {std_wmae:.2f})")

Fold 0: WMAE = 1521.83
🏃 View run XGBoost_Baseline_fold0 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/5f3291a228764a259b6ac161f705c348
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2
Fold 1: WMAE = 1580.92
🏃 View run XGBoost_Baseline_fold1 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/2fdac1a55a4a446abd7c32580091c4eb
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2
Fold 2: WMAE = 1547.43
🏃 View run XGBoost_Baseline_fold2 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/5a0ad543373440c4a9bcc9f9a276c652
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2

Average WMAE across folds: 1550.06 (+/- 24.19)
🏃 View run XGBoost_Baseline at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/700cada9ab3646e196101eb892cb533e
🧪 View experiment at: https://dagshub.com/a

In [ ]:
import random

random.seed(42)

param_grid = {
    "max_depth": [4, 5, 6, 7, 8, 9],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    "n_estimators": [200, 300, 500, 700],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
}

n_trials = 20
results = []

with mlflow.start_run(run_name="XGBoost_HPO_20_trials"):
    mlflow.log_param("n_trials", n_trials)
    mlflow.log_param("search_space", param_grid)

    for trial in range(n_trials):
        trial_params = {k: random.choice(v) for k, v in param_grid.items()}

        with mlflow.start_run(run_name=f"XGBoost_HPO_trial{trial}", nested=True):
            mlflow.log_params(trial_params)

            trial_fold_scores = []
            for X_train, y_train, X_val, y_val, holiday_val in fold_data:
                model = XGBRegressor(
                    enable_categorical=True,
                    tree_method='hist',
                    random_state=42,
                    **trial_params,
                )
                model.fit(X_train, y_train)
                preds = model.predict(X_val)
                trial_fold_scores.append(wmae(y_val.values, preds, holiday_val.values))

            trial_avg_wmae = np.mean(trial_fold_scores)
            trial_std_wmae = np.std(trial_fold_scores)

            mlflow.log_metric("avg_wmae", trial_avg_wmae)
            mlflow.log_metric("std_wmae", trial_std_wmae)

            results.append({**trial_params, "avg_wmae": trial_avg_wmae, "std_wmae": trial_std_wmae})
            print(f"Trial {trial}: avg_wmae={trial_avg_wmae:.2f}, params={trial_params}")

    results_df = pd.DataFrame(results).sort_values("avg_wmae")
    best_params = results_df.iloc[0].drop(["avg_wmae", "std_wmae"]).to_dict()
    best_params['max_depth'] = int(best_params['max_depth'])
    best_params['n_estimators'] = int(best_params['n_estimators'])
    best_params['min_child_weight'] = int(best_params['min_child_weight'])
    best_wmae = results_df.iloc[0]["avg_wmae"]

    mlflow.log_metric("best_wmae", best_wmae)
    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})

    print(f"\nBest trial: avg_wmae={best_wmae:.2f}")
    print(f"Best params: {best_params}")

Trial 0: avg_wmae=2703.21, params={'max_depth': 9, 'learning_rate': 0.01, 'n_estimators': 200, 'subsample': 0.8, 'colsample_bytree': 0.7, 'min_child_weight': 3}
🏃 View run XGBoost_HPO_trial0 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/eacc787603c04a34a39969ff937e72d9
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2
Trial 1: avg_wmae=1674.48, params={'max_depth': 5, 'learning_rate': 0.2, 'n_estimators': 200, 'subsample': 1.0, 'colsample_bytree': 0.6, 'min_child_weight': 7}
🏃 View run XGBoost_HPO_trial1 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/d1dd305a01c74a5bb9196dd3c783f01b
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2
Trial 2: avg_wmae=3605.43, params={'max_depth': 4, 'learning_rate': 0.01, 'n_estimators': 200, 'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_weight': 1}
🏃 View run XGBoost_HPO_trial2 at: https://dagshub.com/aoch

In [ ]:
check_params = {
    'max_depth': 8,
    'learning_rate': 0.03,
    'n_estimators': 700,
    'subsample': 0.7,
    'colsample_bytree': 0.9,
    'min_child_weight': 5,
}

check_scores = []
for i, (X_train, y_train, X_val, y_val, holiday_val) in enumerate(fold_data):
    model = XGBRegressor(
        enable_categorical=True,
        tree_method='hist',
        random_state=42,
        **check_params,
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    fold_wmae = wmae(y_val.values, preds, holiday_val.values)
    check_scores.append(fold_wmae)
    print(f"Fold {i}: WMAE = {fold_wmae:.2f}")

print(f"\nAvg: {np.mean(check_scores):.2f}, Std: {np.std(check_scores):.2f}")

Fold 0: WMAE = 1397.66
Fold 1: WMAE = 1472.77
Fold 2: WMAE = 1458.19

Avg: 1442.88, Std: 32.52


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

class XGBFeaturePrep(BaseEstimator, TransformerMixin):
    def __init__(self, feature_cols):
        self.feature_cols = feature_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['yoy_growth_v2'] = (
            (X['lag_52'] - X['lag_53']) / X['lag_53'].replace(0, np.nan)
        )
        X['Store'] = X['Store'].astype('category')
        X['Dept'] = X['Dept'].astype('category')
        return X[self.feature_cols]

In [ ]:
xgb_pipeline = Pipeline([
    ('feature_prep', XGBFeaturePrep(feature_cols=feature_cols)),
    ('model', XGBRegressor(
        enable_categorical=True,
        tree_method='hist',
        random_state=42,
        **best_params,
    )),
])

X_full_train = processed_train.drop(columns=['Weekly_Sales'])
y_full_train = processed_train['Weekly_Sales']

xgb_pipeline.fit(X_full_train, y_full_train)

Pipeline(steps=[('feature_prep',
                 XGBFeaturePrep(feature_cols=['Store', 'Dept', 'IsHoliday',
                                              'Temperature', 'Fuel_Price',
                                              'MarkDown1', 'MarkDown2',
                                              'MarkDown3', 'MarkDown4',
                                              'MarkDown5', 'CPI',
                                              'Unemployment', 'Size',
                                              'markdown_total',
                                              'n_active_markdowns', 'n_weeks',
                                              'is_sparse', 'week_of_year',
                                              'month', 'year',
                                              'days_to_nearest_holiday',
                                              'is_week_after_thanksgiving',
                                              'l...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.03,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=8, max_leaves=None, min_child_weight=5,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=700,
                              n_jobs=None, num_parallel_tree=None, ...))])

In [ ]:
trusted_types = [
    "__main__.XGBFeaturePrep",
    "xgboost.core.Booster",
    "xgboost.sklearn.XGBRegressor",
]

with mlflow.start_run(run_name="XGBoost_Final"):
    mlflow.log_params(best_params)
    mlflow.log_param("feature_cols", feature_cols)
    mlflow.log_metric("cv_avg_wmae", 1442.88)
    mlflow.log_metric("cv_std_wmae", 32.52)

    mlflow.sklearn.log_model(
        xgb_pipeline,
        artifact_path="xgboost_pipeline",
        skops_trusted_types=trusted_types,
    )

2026/07/09 12:22:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/09 12:22:51 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpjhivrjeg/model/model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.6.1', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 


🏃 View run XGBoost_Final at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2/runs/b8c351d371734c039fecb6b29f515c4b
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/2


In [ ]:
model_id = "m-55ebed8898d94128829ae8f4e111d739"

loaded_pipeline = mlflow.sklearn.load_model(
    f"models:/{model_id}"
)

In [ ]:
test_preds = loaded_pipeline.predict(processed_test)

print(test_preds[:10])
print(f"Any NaNs in predictions: {np.isnan(test_preds).any()}")
print(f"Prediction range: {test_preds.min():.2f} to {test_preds.max():.2f}")
print(f"Number of predictions: {len(test_preds)}")

[43768.69  20704.436 19861.791 20634.64  24395.393 34550.117 43359.598
 47652.516 23757.672 18338.648]
Any NaNs in predictions: False
Prediction range: -6815.22 to 527706.62
Number of predictions: 115064
